# Module 2: Production-Oriented RAG Improvements

## What’s new?

In this module we improve our RAG system by:

- Loading documents from files
- Persisting vector database
- Adding richer metadata
- Returning answers with citations

## Why this matters

These changes move us closer to **enterprise RAG systems**.

In [1]:
%pip install -qU langchain langchain-openai langchain-chroma langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

c:\Users\vammun01\OneDrive - Robert Half\repos\prod-rag\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [5]:
# Initialize environment variables for Azure OpenAI
api_key = (os.getenv("AZURE_OPENAI_API_KEY") or "").strip()
embedding_deployment = (os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME") or "").strip()
endpoint = (os.getenv("AZURE_OPENAI_ENDPOINT") or "").strip()
llm_deployment = (
    os.getenv("AZURE_OPENAI_LLM_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
    or ""
).strip()


# Create embeddings client based on endpoint type
# Foundry project endpoint -> OpenAI-compatible v1 endpoint.
if "services.ai.azure.com" in endpoint and "/api/projects/" in endpoint:
    resource_root = endpoint.split("/api/projects/")[0]
    openai_endpoint = f"{resource_root}/openai/v1"
    embed_model = OpenAIEmbeddings(
        model=embedding_deployment,
        api_key=api_key,
        base_url=openai_endpoint,
    )
    embed_model.embed_query("endpoint-check")
    print("Embeddings client initialized OpenAI-compatible endpoint.")

    llm = ChatOpenAI(
        model=llm_deployment,
        api_key=api_key,
        base_url=openai_endpoint,
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    print("LLM initialized via OpenAI-compatible endpoint.")
else:
    api_version = (os.getenv("AZURE_OPENAI_API_VERSION") or "2024-02-01").strip()
    embed_model = AzureOpenAIEmbeddings(
        model=embedding_deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        openai_api_version=api_version,
    )
    embed_model.embed_query("endpoint-check")
    print(f"Embeddings client initialized via Azure OpenAI endpoint (api_version={api_version}).")

    llm = AzureChatOpenAI(
        azure_deployment=llm_deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=api_version,
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    print(f"LLM initialized via Azure OpenAI endpoint (api_version={api_version}).")
    
    
   

Embeddings client initialized OpenAI-compatible endpoint.
LLM initialized via OpenAI-compatible endpoint.


## Step 1: Simulate File-Based Documents

In real systems, this comes from:
- PDFs
- SharePoint
- S3 / Blob Storage

For now, we simulate file-based input.

In [3]:
documents = [
    Document(
        page_content="LangChain helps build RAG systems using LLMs and retrievers.",
        metadata={"source": "langchain.pdf", "page": 1}
    ),
    Document(
        page_content="LangGraph enables multi-step workflows and agent orchestration.",
        metadata={"source": "langgraph.pdf", "page": 2}
    ),
    Document(
        page_content="LangSmith provides observability and evaluation for LLM systems.",
        metadata={"source": "langsmith.pdf", "page": 3}
    ),
]

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

chunks = text_splitter.split_documents(documents)

print(f"Chunks created: {len(chunks)}")

Chunks created: 3


In [7]:
# Embeddings and Persistent DB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embed_model,
    persist_directory="./chroma_db"
)

## Step 2: Persistent Storage

Unlike Module 1, this vector store is saved locally.

This is important for:
- reuse
- scalability
- production systems

In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [9]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.

Include source citations in your answer.

Context:
{context}

Question:
{question}
""")

In [10]:
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata['source']} (Page {doc.metadata.get('page', 'N/A')})\n{doc.page_content}"
        for doc in docs
    )

In [11]:
def ask_rag(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    
    final_prompt = prompt.invoke({
        "context": context,
        "question": question
    })
    
    response = llm.invoke(final_prompt)
    
    return {
        "answer": response.content,
        "sources": [doc.metadata for doc in docs]
    }

In [12]:
result = ask_rag("What is LangSmith used for?")

print("Answer:\n", result["answer"])
print("\nSources:")
for s in result["sources"]:
    print(s)

Answer:
 LangSmith is used to provide **observability and evaluation for LLM systems**. [langsmith.pdf, p.3]

Sources:
{'source': 'langsmith.pdf', 'page': 3}
{'source': 'langchain.pdf', 'page': 1}
{'page': 2, 'source': 'langgraph.pdf'}
